Environment

In [ ]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [ ]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [ ]:
import os
import json
import copy
import pycomet
import numpy as np
import scipy as sp

from scipy.linalg import expm
from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils

Plotting

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [ ]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [ ]:
sol = h5todict(h5file='./../physics/solution.hdf5')

In [ ]:
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]

In [ ]:
# Compute source term
t, T = [sol[k] for k in ('t', 'T')]
n = np.hstack([na, nm])
omega = kin.function(t=t, y=[n, T])
dnmdt = omega[:,1:-1] / const.UNA
dnmdt = dnmdt.numpy().squeeze()

Analytical solution

In [ ]:
# Solution function
nm_int = lambda t, n0, k: np.matmul(expm(k*t), n0)

In [ ]:
# Construct the evolution matrix
kf, kb = 0.0, 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
    kb = kb + reaction.bwd.k.numpy().squeeze()[0]
k = kb.T + kf
k = (k.T - np.diag(np.sum(k, axis=-1))) * na[0]
# Compute exact solution
nm_anal = np.vstack([nm_int(ti, nm[0], k) for ti in t])
# Compute source term
dnmdt_anal = np.einsum("ij,nj->ni",k,nm_anal) / const.UNA

In [ ]:
# plot_var(
#     x=t.squeeze()[1:],
#     y=nm_anal[1:],
#     labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
#     scales=['log', 'linear'],
#     names=[r'O$_2^{(%d)}$' % (i+1) for i in range(dnmdt.shape[1])],
#     figname=path_to_figs + "/solution"
# )

\begin{align}
    \frac{n_i(t)}{n}
    & =\frac{q_i(t)}{Q} \\
    & =\frac{q_i^\text{eq}f(t)}{Q} \\
    & =\frac{q_i^\text{eq}f(t)}{\sum_iq_i^\text{eq}} \\
    & =\frac{q_i^\text{eq}}{\sum_iq_i^\text{eq}}\exp\left(-\frac{\delta e_i(t)}{k_\text{B}T}\right)
\end{align}

\begin{equation}
    \delta e_i(t) = - k_\text{B}T\log\left(\frac{n_i(t)}{n}\frac{\sum_iq_i^\text{eq}}{q_i^\text{eq}}\right)
\end{equation}

In [ ]:
ng = nm_anal
n = np.sum(ng, axis=-1, keepdims=True)
Qg = sp_th["O2"].pf.int.Q.numpy()
Q = np.sum(Qg, axis=-1, keepdims=True)

In [ ]:
e_eq = sp_th["O2"].thermo.int.e.numpy() * sp_th["O2"].specie.m
de = -const.UKB*T*np.log((ng/n)*(Q/Qg))
e = e_eq + de

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=e[1:]/const.eV_to_J,
    xref=t.squeeze()[1:],
    yref=e_eq[1:]/const.eV_to_J,
    labels=[r'$t$ [s]', r'$e$ [eV]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(de.shape[1])],
    figname=path_to_figs + "/energy"
)

Decoupled Landau-Teller relaxation model
\begin{equation}
    \frac{de_i(t)}{dt}=-\frac{e_i(t)-e_i^{\mathrm{eq}}n_i^{\mathrm{eq}}}{\tau_i}
\end{equation}
with
\begin{equation}
    e_i(t) = \left[e_i^{\mathrm{eq}} + \delta e_i(t)\right]n_i(t)
\end{equation}
and
\begin{equation}
    \tau_i =\frac{f(T\,|\,\theta_i)}{p}
\end{equation}

In [ ]:
e_n = e * ng
e_eq_n = e_eq * ng

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=e_n[1:],
    xref=t.squeeze()[1:],
    yref=e_eq_n[1:],
    labels=[r'$t$ [s]', r'$e$ [J m$^{-3}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(de.shape[1])],
    figname=path_to_figs + "/energy"
)

In [ ]:
idx = np.argmin(np.absolute(e_n - 0.6321*e_eq_n[-1]), axis=0)
tau = [float(t[i+1]) for i in idx]
tau[0] = tau[0] * 1e-2

In [ ]:
de_int = lambda t, de0, tau: expm(-t/tau)*de0

In [ ]:
nb_pts = len(t)
ep = np.zeros_like(e_eq_n)
for i in range(nb_pts):
    for j in range(3):
        de0 = e_n[0,j] - e_eq_n[-1,j]
        ep[i,j] = de_int(t[i,0], de0, tau[j]) + e_eq_n[-1,j]

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=ep[1:],
    xref=t.squeeze()[1:],
    yref=e_n[1:],
    labels=[r'$t$ [s]', r'$e$ [J m$^{-3}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(de.shape[1])],
    figname=path_to_figs + "/energy"
)